In [1]:
"""
#Project structure
#packages to work - wth csv, pdf docs, langchain or other frameworks,streamlit etc.
#refer to requirements.txt file
#analysis using csv file > advanced_analysis
#work with pdf documents > load > splitting >embedding > storing in RAG layer ie vector store 
#build a retriever which extracts info from rag
#building llM appl using fwk such as langchain

--llm defn (llm = model defin)
--scenario = prompt templates + (other options:
     Prompt** (methods to interact with LLM: invoke/chat model)
    --structure
    --templates
    --specific role assignment
    --instructions
    --examples (RAG)
    --output format/template
    --techniques to be used
    --parameter settings/output generation settings
    --request for reasoning/validation in terms of explainations from LLM for the response it generated)
> prompt will use analysis + question to be sent to LLM
--prompt = PromptTemplate(template = scenario,input_variables=['analysis','question'])
--create a chain.. LLMChain(prompt=prompt,llm=llm defn)
--question = 'Look at my analysis and give me a recommendation to improve my sales in every region'
chain.run(analysis, question)


#Developing a chain which will run the tasks sequentially
#1
data_analysis_template = "analyze my data as in {analysis} & provide a initial summary"
prompt_a = PromptTemplate(template = data_analysis_template,input_variables=['analysis'])
create a chain.. 
chain1 = LLMChain(prompt=prompt,llm=llm defn,output = 'output')

#2 

recommendation_template = "based on my {analysis} data & provide a recommendation to addres questions"
prompt_a = PromptTemplate(template = recommendation_template,input_variables=['output','question'])
create a chain.. 
chain2 = LLMChain(prompt=prompt,llm=llm defn,output = 'output2')
chain.run(analysis, question)

overall_chain = SequentialChain(
                chains = [chain1,chain2],
                input_variables = [],
                output_variables = [])
question = "  "
run the overall_chain


#work with pdf documents > load > splitting >embedding > storing in RAG layer ie vector store 
#build a retriever which extracts info from rag
#Implementation of RAG based layer
loaders
splitters > chunking
embedding
stored in vector store
building retriever
  code refer in git for example
     retriever = vectorstore.as_retriever(search_type="similarity", k=4)
     qa_chain = RetrievalQA.from_chain_type(
     llm=llm,
     chain_type='stuff',
     retriever = retriever,
     return_source_documents = True)

#implementation of a tool or call a tool like wikipedia..
#implementation of memory

#evaluation using qaevalchain

#visualization
"""



'\n#Project structure\n#packages to work - wth csv, pdf docs, langchain or other frameworks,streamlit etc.\n#refer to requirements.txt file\n#analysis using csv file > advanced_analysis\n#work with pdf documents > load > splitting >embedding > storing in RAG layer ie vector store \n#build a retriever which extracts info from rag\n#building llM appl using fwk such as langchain\n\n--llm defn (llm = model defin)\n--scenario = prompt templates + (other options:\n     Prompt** (methods to interact with LLM: invoke/chat model)\n    --structure\n    --templates\n    --specific role assignment\n    --instructions\n    --examples (RAG)\n    --output format/template\n    --techniques to be used\n    --parameter settings/output generation settings\n    --request for reasoning/validation in terms of explainations from LLM for the response it generated)\n> prompt will use analysis + question to be sent to LLM\n--prompt = PromptTemplate(template = scenario,input_variables=[\'analysis\',\'question\']

In [2]:
#Setup
#Installation of packages or using a virtual environment with all libraries and packages installed

In [6]:
import pandas as pd

In [7]:
df = pd.read_csv('sales_data.csv')

In [8]:
df.describe
df

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction
0,1/1/2022,Widget C,South,786,26,Male,2.874407
1,1/2/2022,Widget D,East,850,29,Male,3.365205
2,1/3/2022,Widget A,North,871,40,Female,4.547364
3,1/4/2022,Widget C,South,464,31,Male,4.555420
4,1/5/2022,Widget C,South,262,50,Female,3.982935
...,...,...,...,...,...,...,...
2495,10/31/2028,Widget D,North,979,57,Male,3.525510
2496,11/1/2028,Widget D,South,858,30,Female,3.386064
2497,11/2/2028,Widget B,East,878,21,Female,2.272609
2498,11/3/2028,Widget C,South,862,63,Male,2.805692


In [10]:
df.columns
df.describe()

,Sales,Customer_Age,Customer_Satisfaction
count,2500.000000,2500.000000,2500.000000
mean,553.288000,43.332800,3.025869
std,260.101758,14.846758,1.156981
min,100.000000,18.000000,1.005422
25%,324.750000,31.000000,2.056014
50%,552.500000,43.000000,3.049480
75%,779.000000,56.000000,4.042481
max,999.000000,69.000000,4.999006


In [11]:
total_sales = df['Sales'].sum()
avg_sale = df['Sales'].mean()
median_sale = df['Sales'].median()
sales_std = df['Sales'].std()

In [12]:
total_sales,avg_sale,median_sale,sales_std

(1383220, 553.288, 552.5, 260.1017582136852)

In [13]:
df['Month'] = pd.to_datetime(df['Date']).dt.month

In [16]:
#df


In [16]:
monthly_sales = df.groupby('Month', observed=False)['Sales'].sum().sort_values(ascending=False)

In [17]:
monthly_sales

Month
8     124264
5     123646
10    119428
1     118537
7     118493
4     117689
6     115935
3     115367
2     113135
9     111615
11    102775
12    102336
Name: Sales, dtype: int64

In [18]:
best_month = monthly_sales.index[0]

In [19]:
print(best_month)

8


In [17]:
def generate_advanced_data_summary(df):
    # Ensure 'Date' is in datetime format
    df['Date'] = pd.to_datetime(df['Date'])

    # Sales Analysis
    total_sales = df['Sales'].sum()
    avg_sale = df['Sales'].mean()
    median_sale = df['Sales'].median()
    sales_std = df['Sales'].std()

    # Time-based Analysis
    df['Month'] = pd.to_datetime(df['Date']).dt.month
    monthly_sales = df.groupby('Month', observed=False)['Sales'].sum().sort_values(ascending=False)
    best_month = monthly_sales.index[0]
    worst_month = monthly_sales.index[-1]

    # Product Analysis
    product_sales = df.groupby('Product', observed=False)['Sales'].agg(['sum', 'count', 'mean'])
    top_product = product_sales['sum'].idxmax()
    most_sold_product = product_sales['count'].idxmax()

    # Regional Analysis
    region_sales = df.groupby('Region', observed=False)['Sales'].sum().sort_values(ascending=False)
    best_region = region_sales.index[0]
    worst_region = region_sales.index[-1]

    # Customer Analysis
    avg_satisfaction = df['Customer_Satisfaction'].mean()
    satisfaction_std = df['Customer_Satisfaction'].std()

    age_bins = [0, 25, 35, 45, 55, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '55+']
    df['Age_Group'] = pd.cut(df['Customer_Age'], bins=age_bins, labels=age_labels, right=False)
    age_group_sales = df.groupby('Age_Group', observed=False)['Sales'].mean().sort_values(ascending=False)
    best_age_group = age_group_sales.index[0]

    # Gender Analysis
    gender_sales = df.groupby('Customer_Gender', observed=False)['Sales'].mean()

    summary = f"""
    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: ${total_sales:,.2f}
    - Average Sale: ${avg_sale:.2f}
    - Median Sale: ${median_sale:.2f}
    - Sales Standard Deviation: ${sales_std:.2f}

    Time-based Analysis:
    - Best Performing Month: {best_month}
    - Worst Performing Month: {worst_month}

    Product Analysis:
    - Top Selling Product (by value): {top_product}
    - Most Frequently Sold Product: {most_sold_product}

    Regional Performance:
    - Best Performing Region: {best_region}
    - Worst Performing Region: {worst_region}

    Customer Insights:
    - Average Customer Satisfaction: {avg_satisfaction:.2f}/5
    - Customer Satisfaction Standard Deviation: {satisfaction_std:.2f}
    - Best Performing Age Group: {best_age_group}
    - Gender-based Average Sales: Male=${gender_sales['Male']:.2f}, Female=${gender_sales['Female']:.2f}


    Key Observations:
    1. The sales data shows significant variability with a standard deviation of ${sales_std:.2f}.
    2. The {best_age_group} age group shows the highest average sales.
    3. Regional performance varies significantly, with {best_region} outperforming {worst_region}.
    4. The most valuable product ({top_product}) differs from the most frequently sold product ({most_sold_product}), suggesting potential for targeted marketing strategies.
    """

    return summary

In [18]:
print(generate_advanced_data_summary(df))


    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: $1,383,220.00
    - Average Sale: $553.29
    - Median Sale: $552.50
    - Sales Standard Deviation: $260.10

    Time-based Analysis:
    - Best Performing Month: 8
    - Worst Performing Month: 12

    Product Analysis:
    - Top Selling Product (by value): Widget A
    - Most Frequently Sold Product: Widget A

    Regional Performance:
    - Best Performing Region: West
    - Worst Performing Region: East

    Customer Insights:
    - Average Customer Satisfaction: 3.03/5
    - Customer Satisfaction Standard Deviation: 1.16
    - Best Performing Age Group: 18-25
    - Gender-based Average Sales: Male=$547.56, Female=$558.96


    Key Observations:
    1. The sales data shows significant variability with a standard deviation of $260.10.
    2. The 18-25 age group shows the highest average sales.
    3. Regional performance varies significantly, with West outperforming East.
    4. The most valuable produ

In [20]:
summary = generate_advanced_data_summary(df)

In [21]:
print(summary)


    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: $1,383,220.00
    - Average Sale: $553.29
    - Median Sale: $552.50
    - Sales Standard Deviation: $260.10

    Time-based Analysis:
    - Best Performing Month: 8
    - Worst Performing Month: 12

    Product Analysis:
    - Top Selling Product (by value): Widget A
    - Most Frequently Sold Product: Widget A

    Regional Performance:
    - Best Performing Region: West
    - Worst Performing Region: East

    Customer Insights:
    - Average Customer Satisfaction: 3.03/5
    - Customer Satisfaction Standard Deviation: 1.16
    - Best Performing Age Group: 18-25
    - Gender-based Average Sales: Male=$547.56, Female=$558.96


    Key Observations:
    1. The sales data shows significant variability with a standard deviation of $260.10.
    2. The 18-25 age group shows the highest average sales.
    3. Regional performance varies significantly, with West outperforming East.
    4. The most valuable produ

In [25]:
import langchain
import langchain_core
import langchain_community
print(langchain.__version__)
print(langchain_core.__version__)
print(langchain_community.__version__)

1.2.3
1.2.7
0.4.1


In [26]:
# Chains
#from langchain.chains import LLMChain, SimpleSequentialChain --for older versions
from langchain_core.runnables import RunnableSequence

# Models
from langchain_openai import ChatOpenAI, OpenAI, OpenAIEmbeddings

# Documents
from langchain_core.documents import Document

# Prompts
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
)

# Messages
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
)

# Vector stores
from langchain_community.vectorstores import Chroma

# Text splitters
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    SentenceTransformersTokenTextSplitter,
)


In [27]:
from langchain_openai import AzureChatOpenAI

In [30]:
import os
from dotenv import load_dotenv
load_dotenv("E:\\Lesson_2_demos\\.env")

## Start by creating an instance of the AzureChatOpenAI class.
client = AzureChatOpenAI(
     azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
     api_key=os.getenv("API_KEY"),
     api_version="2024-12-01-preview",
     deployment_name="gpt-4.1",
     temperature=0,
 )

#Other variant
#llm = AzureChatOpenAI(
#    azure_deployment="gpt-4.1",         # deployment name from Azure portal
#    azure_endpoint="myendpoint",           # your endpoint
#    api_version="2023-12-01-preview",
#    api_key="mykey",
#    temperature=0.0
#)

In [33]:
template = """Question: {question}
Answer: Let's think step by step and generate one line answers and no metatdata is needed in response"""
prompt = PromptTemplate(template=template, input_variables=["question"])

# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

In [34]:
# # RunnableSequence
chain = prompt | client

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))


Q: Explain the concept of black holes in simple terms.
content='A black hole is a place in space where gravity is so strong that nothing, not even light, can escape from it.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 40, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_63d8723934', 'id': 'chatcmpl-DEHeSbfG7PvlxBFydvdusvGRxyDeg', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'

In [ ]:
#If using openai
# from openai import AzureOpenAI

# from dotenv import load_dotenv
# load_dotenv("E:\\Lesson_2_demos\\.env")

# client = AzureOpenAI(
#     api_key=os.getenv("API_KEY"),
#     api_version="2024-12-01-preview",
#     azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
# )


In [35]:
client

AzureChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027595266DD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000275A933A440>, root_client=<openai.lib.azure.AzureOpenAI object at 0x000002758A25BE80>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x00000275A8DD7580>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://aiwrk.cognitiveservices.azure.com/', deployment_name='

In [36]:
summary

'\n    Advanced Sales Data Summary:\n\n    Overall Sales Metrics:\n    - Total Sales: $1,383,220.00\n    - Average Sale: $553.29\n    - Median Sale: $552.50\n    - Sales Standard Deviation: $260.10\n\n    Time-based Analysis:\n    - Best Performing Month: 8\n    - Worst Performing Month: 12\n\n    Product Analysis:\n    - Top Selling Product (by value): Widget A\n    - Most Frequently Sold Product: Widget A\n\n    Regional Performance:\n    - Best Performing Region: West\n    - Worst Performing Region: East\n\n    Customer Insights:\n    - Average Customer Satisfaction: 3.03/5\n    - Customer Satisfaction Standard Deviation: 1.16\n    - Best Performing Age Group: 18-25\n    - Gender-based Average Sales: Male=$547.56, Female=$558.96\n\n\n    Key Observations:\n    1. The sales data shows significant variability with a standard deviation of $260.10.\n    2. The 18-25 age group shows the highest average sales.\n    3. Regional performance varies significantly, with West outperforming East

In [44]:
scenario_template = """
You are an AI sales analyst. Use the summary data which we have analyzed, provide some in-depth analysis and some recommendation based on 
data provided. Be specific to the data points and give me citations if the recommendations were based on some web or public resources.
Also give me 2 additional lines on where was similar scenario observed in past in real world.
{summary}
Question: {question}

Detaled Analysis & recommendation by AI:
"""

In [45]:
prompt = PromptTemplate(template=scenario_template,input_variables=["summary","question"])

In [47]:
#newer version
chain = prompt | client
question = "based on this data, what are our areas of improvement"
def gen_insight(summary, question):
    return chain.invoke({"summary": summary, "question": question}).content

In [48]:
insight = gen_insight(summary,question)
print(insight)

**Detailed Analysis & Recommendations**

### 1. Sales Variability & Consistency

**Analysis:**  
- The sales standard deviation ($260.10) is nearly half the average sale ($553.29), indicating high variability in sales amounts. This suggests inconsistent sales performance across transactions, possibly due to product mix, pricing, or customer segments.

**Recommendation:**  
- **Standardize Sales Processes:** Implement more consistent sales strategies, such as bundling products or offering tiered pricing. This can help reduce variability and increase predictability in sales.  
- **Focus on High-Value Customers:** Segment customers based on purchase history and target those with higher average sales for upselling and loyalty programs.  
- **Citation:** McKinsey’s research on sales process standardization shows that companies with consistent sales processes outperform peers by up to 28% in revenue growth ([McKinsey, 2022](https://www.mckinsey.com/capabilities/growth-marketing-and-sales/our

In [49]:
#from langchain import SequentialChain
##implementation of seq chain to do the same thing as above in 2 different chains followed
#sequentially

#1st
#data_analysis_template = """analyze my {analysis} data  and provide a summary """
#a_prompt = PromptTemplate(template=data_analysis_template,input_variables=["analysis"])
#chain1 = LLMChain(prompt=a_prompt,llm=myllm,output='output')

#2nd
#recommendation_template = """based on {analysis} data  and provide a recommendaton to address the question: {question} """
#b_prompt = PromptTemplate(template=recommendation_template,input_variables=["analysis","question"])
#chain2 = LLMChain(prompt=b_prompt,llm=myllm,output='output2')

'''overall_chain = SequentialChain(
       chains = [chain1,chain2],
       input_variables=['summary','question'],
       output_variables = ['analysis','recommendation'],
)'''

"overall_chain = SequentialChain(\n       chains = [chain1,chain2],\n       input_variables=['summary','question'],\n       output_variables = ['analysis','recommendation'],\n)"

In [55]:
'''def gen_insight(question):
    result = overall_chain.run({"summary": summary,"question": question})
    return f"Analysis:\n{result['analysis']\n\nRecommendation:\n{result['recommendation']}"'''

'def gen_insight(question):\n    result = overall_chain.run({"summary": summary,"question": question})\n    return f"Analysis:\n{result[\'analysis\']\n\nRecommendation:\n{result[\'recommendation\']}"'

In [56]:
'''question = "how can we improve in a region"'''
'''questions = ['how can we improve in a region','what would be better approach to cater to people from
                a specific region']
   for i in questions:
       print(gen_insight(i))'''
'''insight = gen_insight(question)
print(insight)'''

'insight = gen_insight(question)\nprint(insight)'

In [61]:
#from langchain_core.documents import PyPDFLoader

In [60]:
!pip install pypdf


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
import pypdf

In [63]:
#from langchain.text_splitter import RecursiveCharacterTextSplitter

In [71]:
import os
os.listdir('PDF Folder')

['AI business model innovation.pdf',
 'BI approaches.pdf',
 'Time-Series-Data-Prediction-using-IoT-and-Machine-Le_2020_Procedia-Computer-.pdf',
 'Walmarts sales data analysis.pdf']

In [72]:
pdf_folder = 'PDF Folder'

In [73]:
from pypdf import PdfReader

In [75]:
documents = []
for file in os.listdir(pdf_folder):
    if file.endswith('.pdf'):
        print(file)

AI business model innovation.pdf
BI approaches.pdf
Time-Series-Data-Prediction-using-IoT-and-Machine-Le_2020_Procedia-Computer-.pdf
Walmarts sales data analysis.pdf


In [78]:
#code to be relooked..
documents = []
for file in os.listdir(pdf_folder):
    if file.endswith('.pdf'):
        with open(file, 'rb') as pdf_file:
        # Create a PdfReader object
            reader = PdfReader(pdf_file)
        # Access document information
            print(f"Total pages: {len(reader.pages)}")
            metadata = reader.metadata
            if metadata:
                print(f"Author: {metadata.author}")
                print(f"Title: {metadata.title}")
        
            # Extract text from the first page (optional)
            first_page = reader.pages[0]
            text = first_page.extract_text()
            print(text)


FileNotFoundError: [Errno 2] No such file or directory: 'AI business model innovation.pdf'

In [51]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap= 200)

In [52]:
texts = text_splitter.split_documents(documents)

In [75]:
#optional
#save your texts as .pkl

In [53]:
from langchain_community.embeddings import HuggingFaceEmbeddings
model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model)

C:\Users\Ajay\AppData\Local\Temp\ipykernel_10556\3834499437.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model)


In [54]:
!pip install faiss-cpu


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from langchain.vectorstores import FAISS

In [56]:
from langchain.chains import RetrievalQA

In [80]:
#from langchain.utilities import WikipediaAPIWrapper
#from datetime import datetime

In [57]:
vectorstore = FAISS.from_documents(texts, embeddings)

In [59]:
#Testing

'''all_docs = list(vectorstore.docstore._dict.values())

for i, doc in enumerate(all_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")'''

'all_docs = list(vectorstore.docstore._dict.values())\n\nfor i, doc in enumerate(all_docs):\n    print(f"Document {i+1}:\n{doc.page_content}\n")'

In [61]:
retriever = vectorstore.as_retriever(search_type="similarity", k=4)

In [62]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [87]:
#define a function to use wiki search

In [63]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm = llm,
    memory = memory,
    verbose = True
)

C:\Users\Ajay\AppData\Local\Temp\ipykernel_10556\3326585183.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
C:\Users\Ajay\AppData\Local\Temp\ipykernel_10556\3326585183.py:5: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


In [64]:
question = 'what are top selling products'

In [66]:
context = f"my summary data is:\n{summary}\nQuestion:{question}"

In [67]:
result = qa_chain({"query": context})

C:\Users\Ajay\AppData\Local\Temp\ipykernel_10556\431315255.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": context})


In [68]:
result

{'query': 'my summary data is:\n\n    Advanced Sales Data Summary:\n\n    Overall Sales Metrics:\n    - Total Sales: $1,383,220.00\n    - Average Sale: $553.29\n    - Median Sale: $552.50\n    - Sales Standard Deviation: $260.10\n\n    Time-based Analysis:\n    - Best Performing Month: 8\n    - Worst Performing Month: 12\n\n    Product Analysis:\n    - Top Selling Product (by value): Widget A\n    - Most Frequently Sold Product: Widget A\n\n    Regional Performance:\n    - Best Performing Region: West\n    - Worst Performing Region: East\n\n    Customer Insights:\n    - Average Customer Satisfaction: 3.03/5\n    - Customer Satisfaction Standard Deviation: 1.16\n    - Best Performing Age Group: 18-25\n    - Gender-based Average Sales: Male=$547.56, Female=$558.96\n\n\n    Key Observations:\n    1. The sales data shows significant variability with a standard deviation of $260.10.\n    2. The 18-25 age group shows the highest average sales.\n    3. Regional performance varies significantl

In [94]:
#get wiki content related to question

In [69]:
conversation.predict(input=f"my summary data is:\n{summary}\nQuestion:{question}")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: my summary data is:

    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: $1,383,220.00
    - Average Sale: $553.29
    - Median Sale: $552.50
    - Sales Standard Deviation: $260.10

    Time-based Analysis:
    - Best Performing Month: 8
    - Worst Performing Month: 12

    Product Analysis:
    - Top Selling Product (by value): Widget A
    - Most Frequently Sold Product: Widget A

    Regional Performance:
    - Best Performing Region: West
    - Worst Performing Region: East

    Customer Insights:
    - Average Customer Satisfaction: 3.03/5
    - Customer Satisfaction Standard Deviation: 1.16
    - Best Performing Age Group: 18

'According to your summary data, the top selling product by value is **Widget A**. Additionally, Widget A is also the most frequently sold product. This means Widget A leads both in total sales revenue and in the number of units sold, making it the clear top performer among your products. If you have data on other products, I could help analyze how they compare or suggest strategies to boost sales of other items!'

In [70]:
def insights_based_on_memory(question):
    return conversation.predict(input=f"my summary data is:\n{summary}\nQuestion:{question}")

In [98]:
insights_based_on_memory("which region is performing better and why")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: my summary data is:

    Advanced Sales Data Summary:

    Overall Sales Metrics:
    - Total Sales: $1,383,220.00
    - Average Sale: $553.29
    - Median Sale: $552.50
    - Sales Standard Deviation: $260.10

    Time-based Analysis:
    - Best Performing Month: 8
    - Worst Performing Month: 12

    Product Analysis:
    - Top Selling Product (by value): Widget A
    - Most Frequently Sold Product: Widget A

    Regional Performance:
    - Best Performing Region: West
    - Worst Performing Region: East

    Customer Insights:
    - Average Customer Satisfaction: 3.03/5
    - Customer Satisfaction Standard Deviation: 1.16
    - Best Performing Age Group: 18-

"The best performing region in your sales data is the West, while the worst performing region is the East. Although the summary doesn't provide specific reasons for this regional performance difference, several factors could contribute:\n\n1. **Market Demand:** The West region might have higher demand for your products, possibly due to demographic factors, economic conditions, or consumer preferences aligning better with your offerings.\n\n2. **Sales and Marketing Efforts:** There could be stronger or more effective sales and marketing campaigns in the West, leading to higher sales volumes and values.\n\n3. **Distribution and Availability:** Better distribution networks or product availability in the West might make it easier for customers to purchase your products.\n\n4. **Customer Satisfaction and Engagement:** While the overall average customer satisfaction is 3.03/5, regional variations might exist, with the West possibly having higher satisfaction, encouraging repeat purchases.\n\

In [ ]:
#QAEvalChain